# REHAB24-6 check: MediaPipe elbow angle vs motion capture (arm exercises only)

Uses **your repo** (`suvini-weel02`: `angles.py`, `visibility.py`, `filters.py`, `pose/landmarks.py`) and only the **arm recordings**
(Ex1 arm abduction, Ex2 arm VW, Ex3 push-ups on a table) from the [REHAB24-6 dataset](https://zenodo.org/records/13305826).
Both arms, both cameras, every labelled repetition window.

**Run the cells in order.** There are three STOP points where you check the printout before going on:
1. After the install cell: restart the session.
2. After the inventory cell: do the file names and `Segmentation.csv` columns look right?
3. After the pairing cell: does each video match the right joints file?

Start with `MAX_VIDEOS = 2` (a quick test), then set it to `None` for the full run.
The dataset page does not describe its file formats, so the file-finding cells guess and print what they guessed. If a guess is wrong, paste me the output.

## 1. Install

Your repo uses the older MediaPipe (`mp.solutions.pose`), which only exists for Python 3.12 or older, and Colab may run a newer Python.
So MediaPipe runs inside its own small Python 3.12 environment (made with `uv`), and the analysis runs in Colab's normal Python.
No restart is needed. This cell takes a few minutes.

In [ ]:
!pip -q install uv
!uv venv --python 3.12 /content/mpenv
!uv pip install --python /content/mpenv/bin/python "numpy==1.26.4" "mediapipe==0.10.14" pyyaml

In [ ]:
import subprocess
VENV_PY = "/content/mpenv/bin/python"
r = subprocess.run([VENV_PY, "-c", "import sys, numpy, mediapipe as mp, cv2; print(sys.version.split()[0], numpy.__version__, mp.__version__, hasattr(mp.solutions, 'pose'))"],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)
assert r.returncode == 0 and r.stdout.strip().endswith("True"), "MediaPipe environment failed. Paste this output to Claude."
import cv2, numpy as np    # the analysis side (Colab's own Python)

## 2. Get your code and set the options

In [ ]:
import os, sys, re, time, glob, shutil, subprocess, zipfile
REPO_DIR = "/content/repo"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", "-b", "suvini-weel02",
                    "https://github.com/suvini2001/AI-based-Physiotherapy-project-.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
for p in (".", "capture", "pose"):
    sys.path.append(os.path.abspath(p))

# ---------------- options ----------------
MAX_VIDEOS   = 2        # 2 = quick test. Set to None for ALL arm recordings.
EXERCISES    = [1, 2, 3]  # Ex1 arm abduction, Ex2 arm VW, Ex3 push-ups on table
THRESHOLD    = 0.70     # your chosen visibility threshold
MARGIN       = 45       # extra frames before/after the labelled repetitions (needed to find the time shift)
MAX_LAG      = 30       # largest time shift searched (frames)
MODEL_COMPLEXITY = 1    # same as your landmarks.py default
USE_DRIVE    = False    # True = keep landmarks/results on Google Drive so a reset does not lose them
# If the automatic guess for the arm joints fails, fill these with names from joints_names.txt:
ARMS_OVERRIDE = {}      # e.g. {"right": ("RShoulder", "RElbow", "RWrist"), "left": (...)}
# If the Segmentation.csv column guess is wrong, name the columns here:
COL = {"video": None, "exercise": None, "start": None, "end": None}

DATA, CACHE, RESULTS = "/content/rehab24", "/content/cache", "/content/results"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE, RESULTS = "/content/drive/MyDrive/rehab24_cache", "/content/drive/MyDrive/rehab24_results"
for d in (DATA, CACHE, RESULTS):
    os.makedirs(d, exist_ok=True)

# small script that runs YOUR LandmarkExtractor on one frame window, inside the MediaPipe environment
open(f"{REPO_DIR}/extract_window.py", "w").write("""
import sys, numpy as np, cv2
sys.path.append("pose")
from landmarks import LandmarkExtractor
video, f0, f1, out, cx = sys.argv[1], int(sys.argv[2]), int(sys.argv[3]), sys.argv[4], int(sys.argv[5])
cap = cv2.VideoCapture(video)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
ex, rows, size, i = LandmarkExtractor(cx), [], None, 0
while i <= f1:
    if i < f0:
        if not cap.grab(): break
    else:
        ok, frame = cap.read()
        if not ok: break
        size = size or (frame.shape[1], frame.shape[0])
        rows.append(ex.process(frame))
    i += 1
ex.close(); cap.release()
if not rows: raise SystemExit("no frames read from " + video)
np.savez(out, lm=np.stack(rows), size=np.array(size), fps=fps)
""")

## 3. Download and unzip (only the 2D data and videos; 3D is not needed here)

In [ ]:
BASE = "https://zenodo.org/records/13305826/files"

def fetch(name):
    out = f"{DATA}/{name}"
    if os.path.exists(out) and os.path.getsize(out) > 0:
        print("already have", name); return True
    subprocess.run(["wget", "-nv", "-c", "-O", out, f"{BASE}/{name}?download=1"])
    ok = os.path.exists(out) and os.path.getsize(out) > 0
    print(("downloaded " if ok else "FAILED     ") + name, f"{os.path.getsize(out)/1e6:.1f} MB" if ok else "")
    if not ok and os.path.exists(out): os.remove(out)
    return ok

for n in ("Segmentation.csv", "Segmentation.txt", "2d_joints.zip", "videos.zip"):
    fetch(n)
if not fetch("joints_names.txt"):
    fetch("joint_names.txt")      # the Zenodo page spells it both ways

def unzip(zname, folder):
    z, out = f"{DATA}/{zname}", f"{DATA}/{folder}"
    if os.path.isdir(out) and os.listdir(out):
        print("already unzipped", folder); return
    os.makedirs(out, exist_ok=True)
    if shutil.which("unzip"):
        subprocess.run(["unzip", "-q", "-o", z, "-d", out], check=True)
    else:
        zipfile.ZipFile(z).extractall(out)
    print("unzipped", folder)

unzip("videos.zip", "videos")
unzip("2d_joints.zip", "2d_joints")

## 4. Inventory  (STOP: read this output)

In [ ]:
import pandas as pd
from collections import Counter

def files_under(folder, exts):
    return sorted(p for p in glob.glob(f"{DATA}/{folder}/**/*", recursive=True) if p.lower().endswith(exts))

VIDEOS = files_under("videos", (".mp4", ".avi", ".mov", ".mkv"))
JOINTS = files_under("2d_joints", (".npy", ".csv", ".txt", ".npz"))
print(len(VIDEOS), "videos,", len(JOINTS), "2D joint files")
print("video examples :", [os.path.relpath(p, DATA) for p in VIDEOS[:4]])
print("joints examples:", [os.path.relpath(p, DATA) for p in JOINTS[:4]])

NAMES_FILE = next(p for p in (f"{DATA}/joints_names.txt", f"{DATA}/joint_names.txt") if os.path.exists(p))
NAMES = [l.strip() for l in open(NAMES_FILE, encoding="utf-8", errors="ignore") if l.strip()]
print(f"\n{len(NAMES)} joint names:", NAMES)

raw = open(f"{DATA}/Segmentation.csv", encoding="utf-8", errors="ignore").read(2000)
seg = pd.read_csv(f"{DATA}/Segmentation.csv", sep=None, engine="python")
print("\nSegmentation.csv columns:", list(seg.columns), "| rows:", len(seg))
print(seg.head(5).to_string())
print("\n--- Segmentation.txt ---")
print(open(f"{DATA}/Segmentation.txt", encoding="utf-8", errors="ignore").read()[:1500])
a = np.load(JOINTS[0], allow_pickle=True) if JOINTS[0].endswith(".npy") else None
if a is not None:
    print("\nfirst joints file shape:", a.shape, a.dtype, "| first values:", np.round(np.ravel(a)[:6], 1))

## 5. Find the arm joints and pair each video with its joints file  (STOP: check the table)

In [ ]:
# --- arm joint names ---
def find_arm(names, side):
    s = side[0]                     # 'r' or 'l'
    out = []
    for kind in ("shoulder", "elbow", "wrist"):
        hits = []
        for i, n in enumerate(names):
            low = n.lower()
            if kind not in low and not (kind == "shoulder" and "sho" in low) and not (kind == "wrist" and "wri" in low):
                continue
            if low.startswith(side) or low.startswith(s) or side in low or re.search(rf"(^|[^a-z]){s}([^a-z]|{kind[:3]})", low):
                hits.append(i)
        if len(hits) != 1:
            raise SystemExit(f"Could not pick the {side} {kind} from {names}. Fill ARMS_OVERRIDE in the options cell.")
        out.append(names[hits[0]])
    return tuple(out)

ARMS = {side: ARMS_OVERRIDE.get(side) or find_arm(NAMES, side) for side in ("right", "left")}
print("arm joints:", ARMS)

# --- file keys: recording number(s) + camera ---
STOP_WORDS = {"joints", "joint", "2d", "3d", "video", "videos", "fps", "markers", "npy", "mp4", "csv"}
def file_key(path_or_text):
    stem = os.path.splitext(os.path.basename(str(path_or_text)))[0].lower()
    cam = re.search(r"(?:cam(?:era)?|c)[_\-]?(17|18)", stem)
    s = re.sub(r"(?:cam(?:era)?|c)[_\-]?(17|18)", " ", stem)
    s = re.sub(r"\d+\s*fps", " ", s)
    digits = tuple(int(x) for x in re.findall(r"\d+", s))
    alpha = tuple(t for t in re.findall(r"[a-z]+", s) if t not in STOP_WORDS)
    return alpha, digits, (cam.group(1) if cam else None)

def same_recording(k1, k2):
    return k1[1] == k2[1] and (not k1[0] or not k2[0] or k1[0] == k2[0]) and (k1[2] is None or k2[2] is None or k1[2] == k2[2])

# --- Segmentation columns ---
def guess(keys):
    for k in keys:
        for c in seg.columns:
            if k in str(c).lower():
                return c
cvid = COL["video"] or guess(["video", "recording", "record", "sequence", "file", "id"])
cex  = COL["exercise"] or guess(["exercise", "ex"])
cst  = COL["start"] or guess(["first", "start", "begin"])
cen  = COL["end"] or guess(["last", "end", "stop"])
print("Segmentation columns used -> video:", cvid, "| exercise:", cex, "| start:", cst, "| end:", cen)
assert None not in (cvid, cex, cst, cen), "Fill COL in the options cell with the right column names (see the columns printed above)."

seg["_ex"] = seg[cex].astype(str).str.extract(r"(\d+)")[0].astype(float)
arm_rows = seg[seg["_ex"].isin(EXERCISES)].copy()
print("arm repetitions:", len(arm_rows), "of", len(seg), "| per exercise:", arm_rows["_ex"].value_counts().sort_index().to_dict())

# --- pair: one record per video x labelled recording ---
recs = []
vkeys = {p: file_key(p) for p in VIDEOS}
jkeys = {p: file_key(p) for p in JOINTS}
for vid_value, g in arm_rows.groupby(cvid):
    rk = file_key(str(vid_value))
    ex = int(g["_ex"].iloc[0])
    f0, f1 = int(g[cst].min()), int(g[cen].max())
    for vp, vk in vkeys.items():
        if not same_recording(rk, vk):
            continue
        js = [jp for jp, jk in jkeys.items() if same_recording(vk, jk) and jk[2] == vk[2]] or \
             [jp for jp, jk in jkeys.items() if same_recording(vk, jk)]
        if len(js) == 1:
            recs.append(dict(name=os.path.splitext(os.path.basename(vp))[0], video=vp, joints=js[0],
                             cam=vk[2] or "?", ex=ex, f0=f0, f1=f1))
        else:
            print("  could not pair uniquely:", os.path.basename(vp), "->", [os.path.basename(j) for j in js][:3])

recs.sort(key=lambda r: (r["ex"], r["name"]))
table = pd.DataFrame(recs)
print(f"\n{len(recs)} video/joint pairs for exercises {EXERCISES}")
print(table.assign(video=table.video.map(os.path.basename), joints=table.joints.map(os.path.basename)).head(12).to_string()
      if len(recs) else "NO PAIRS FOUND")
assert recs, "No pairs found. Paste this cell's output to Claude."

**STOP:** read the table. Each `video` must sit next to the `joints` file of the *same* recording and camera.
If it does not, paste the output to Claude. If it looks right, continue.

## 6. Helper functions (uses your `angles.py`, `visibility.py`, `filters.py`, `landmarks.py`)

In [ ]:
from angles import angle, joint_angle, ELBOW
from filters import butterworth_filter, fill_gaps, rolling_median_filter, savitzky_golay_filter
from visibility import mask_low_visibility

# ---------- truth ----------
def load_truth(path):
    if path.endswith(".npy"):
        arr = np.load(path, allow_pickle=True)
    elif path.endswith(".npz"):
        z = np.load(path); arr = z[z.files[0]]
    else:
        arr = np.genfromtxt(path, delimiter="," if "," in open(path).readline() else None)
    arr = np.asarray(arr, float)
    nj = len(NAMES)
    if arr.ndim == 2:
        arr = arr.reshape(len(arr), nj, -1)
    if arr.ndim == 3 and arr.shape[1] != nj and arr.shape[0] == nj:
        arr = arr.transpose(1, 0, 2)
    assert arr.ndim == 3 and arr.shape[1] == nj, f"unexpected joints shape {arr.shape} (expected frames x {nj} joints x dims)"
    return arr

def truth_elbow(arr, side):
    idx = [NAMES.index(n) for n in ARMS[side]]
    return angle(arr[:, idx[0], :2], arr[:, idx[1], :2], arr[:, idx[2], :2])    # 2D, like a single camera

# ---------- MediaPipe on a frame window (cached) ----------
def get_landmarks(rec):
    f0w, f1w = rec["f0w"], rec["f1w"]
    cache = f"{CACHE}/{rec['name']}_{f0w}_{f1w}.npz"
    if not os.path.exists(cache):
        r = subprocess.run([VENV_PY, "extract_window.py", rec["video"], str(f0w), str(f1w), cache, str(MODEL_COMPLEXITY)],
                           capture_output=True, text=True, cwd=REPO_DIR)
        if r.returncode != 0:
            raise RuntimeError("MediaPipe step failed:\n" + r.stderr[-1500:])
    z = np.load(cache)
    return z["lm"], tuple(int(v) for v in z["size"]), float(z["fps"])

def video_frames(path):
    cap = cv2.VideoCapture(path); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release(); return n

# ---------- alignment + metrics ----------
def overlap(a, b, lag):                      # pairs a[i] with b[i+lag]
    if lag >= 0:
        n = min(len(a), len(b) - lag); return a[:n], b[lag:lag + n]
    n = min(len(a) + lag, len(b)); return a[-lag:-lag + n], b[:n]

def best_lag(mp_ang, truth_ang, max_lag):
    prep = lambda x: (lambda y: y - y.mean())(fill_gaps(x.copy()))
    a, b = prep(mp_ang), prep(truth_ang)
    best, best_r = 0, -2
    for lag in range(-max_lag, max_lag + 1):
        x, y = overlap(a, b, lag)
        if len(x) < 10 or x.std() == 0 or y.std() == 0: continue
        r = np.corrcoef(x, y)[0, 1]
        if r > best_r: best, best_r = lag, r
    return best, best_r

def icc_2_1(x, y):
    d = np.column_stack([x, y]); n, k = d.shape; gm = d.mean()
    msr = k * ((d.mean(1) - gm) ** 2).sum() / (n - 1)
    msc = n * ((d.mean(0) - gm) ** 2).sum() / (k - 1)
    mse = ((d - d.mean(1, keepdims=True) - d.mean(0, keepdims=True) + gm) ** 2).sum() / ((n - 1) * (k - 1))
    return (msr - mse) / (msr + (k - 1) * mse + k * (msc - mse) / n)

def metrics(mp_ang, truth_ang):
    ok = ~(np.isnan(mp_ang) | np.isnan(truth_ang))
    x, y = mp_ang[ok], truth_ang[ok]
    if len(x) < 10: return None
    d = x - y
    return dict(n=len(x), coverage=100 * ok.sum() / len(mp_ang), mae=np.abs(d).mean(), rmse=np.sqrt((d ** 2).mean()),
                bias=d.mean(), loa_lo=d.mean() - 1.96 * d.std(ddof=1), loa_hi=d.mean() + 1.96 * d.std(ddof=1),
                r=np.corrcoef(x, y)[0, 1], icc=icc_2_1(x, y))

FILTERS = ["none", "median5+savgol", "butter6 offline", "butter6 causal"]
def apply_filter(ang, name, fs):
    nan = np.isnan(ang)
    if nan.all() or name == "none": return ang
    x = fill_gaps(ang.copy())
    if name == "median5+savgol": y = savitzky_golay_filter(rolling_median_filter(x, 5), 9, 3)
    elif name == "butter6 offline": y = butterworth_filter(x, fs, 6.0)
    else: y = butterworth_filter(x, fs, 6.0, causal=True)
    y = np.clip(y, 0, 180); y[nan] = np.nan
    return y

THRESHOLDS = (0.5, 0.6, 0.7, 0.8, 0.9)

## 7. Run it  (the slow cell: MediaPipe on every arm window; results are cached)

In [ ]:
rows, store = [], {}
todo = recs[:MAX_VIDEOS] if MAX_VIDEOS else recs
t0 = time.time()
for n, rec in enumerate(todo, 1):
    arr = load_truth(rec["joints"])
    nv = video_frames(rec["video"])
    if nv and abs(len(arr) / nv - 1) > 0.02:                       # mocap and video lengths differ -> resample truth by index
        print(f"  note: {rec['name']}: {len(arr)} truth frames vs {nv} video frames; resampling truth")
        arr = arr[np.round(np.linspace(0, len(arr) - 1, nv)).astype(int)]
    rec["f0w"], rec["f1w"] = max(0, rec["f0"] - MARGIN), min(len(arr) - 1, rec["f1"] + MARGIN)
    lm, size, fps = get_landmarks(rec)
    tl = {s: truth_elbow(arr, s)[rec["f0w"]: rec["f0w"] + len(lm)] for s in ("right", "left")}
    for side in ("right", "left"):
        other = "left" if side == "right" else "right"
        mp_at = lambda th: joint_angle(mask_low_visibility(lm, th), ELBOW[side], size)
        base = mp_at(THRESHOLD)
        lag, r_named = best_lag(base, tl[side], MAX_LAG)
        lag_o, r_other = best_lag(base, tl[other], MAX_LAG)
        swapped = r_other > r_named + 0.1                            # left/right mirrored between video and mocap?
        truth_ang, lag, r_used = (tl[other], lag_o, r_other) if swapped else (tl[side], lag, r_named)
        pair = lambda x: overlap(x, truth_ang, lag)
        x, y = pair(base)
        m = metrics(x, y)
        if m is None: continue
        rows.append(dict(video=rec["name"], cam=rec["cam"], ex=rec["ex"], side=side, swapped=swapped, lag=lag,
                         truth_min=np.nanmin(y), truth_max=np.nanmax(y), mp_min=np.nanmin(x), mp_max=np.nanmax(x), **m))
        store[(rec["name"], side)] = dict(
            ex=rec["ex"], cam=rec["cam"], truth=y, base=x,
            thr={th: pair(mp_at(th))[0] for th in THRESHOLDS},
            filt={f: pair(apply_filter(base, f, fps))[0] for f in FILTERS})
    print(f"[{n}/{len(todo)}] {rec['name']} (Ex{rec['ex']}, cam {rec['cam']}) done, {time.time()-t0:.0f}s elapsed")

res = pd.DataFrame(rows)
res.to_csv(f"{RESULTS}/per_recording.csv", index=False)
print(res.round(2).to_string())

## 8. Results

In [ ]:
import matplotlib.pyplot as plt

def pooled(keys, getter):
    xs = np.concatenate([getter(store[k])[0] for k in keys]); ys = np.concatenate([getter(store[k])[1] for k in keys])
    return metrics(xs, ys)

def fmt(m):
    return (f"n={m['n']}  MAE={m['mae']:.1f}  RMSE={m['rmse']:.1f}  bias={m['bias']:+.1f}  "
            f"LoA=[{m['loa_lo']:+.1f},{m['loa_hi']:+.1f}]  r={m['r']:.2f}  ICC={m['icc']:.2f}")

keys = list(store)
print("== Per exercise x camera: mean over recordings ==")
summ = res.groupby(["ex", "cam"])[["mae", "rmse", "bias", "r", "icc", "coverage", "truth_min", "truth_max"]].mean().round(2)
summ["recordings"] = res.groupby(["ex", "cam"]).size()
print(summ.to_string()); summ.to_csv(f"{RESULTS}/summary_ex_cam.csv")

print("\n== Pooled over all frames (threshold %.2f, no filter) ==" % THRESHOLD)
print("ALL      ", fmt(pooled(keys, lambda s: (s["base"], s["truth"]))))
for ex in sorted({store[k]["ex"] for k in keys}):
    ks = [k for k in keys if store[k]["ex"] == ex]
    print(f"Ex{ex}      ", fmt(pooled(ks, lambda s: (s["base"], s["truth"]))))
print("recordings where left/right looked swapped:", int(res.swapped.sum()), "of", len(res),
      "| time shifts (frames): median", res.lag.median(), ", range", res.lag.min(), "to", res.lag.max())
print("low-agreement rows (r < 0.5):", int((res.r < 0.5).sum()), "of", len(res))

print("\n== Visibility threshold sweep (pooled, no filter) ==")
thr_rows = []
for th in THRESHOLDS:
    m = pooled(keys, lambda s: (s["thr"][th], s["truth"])); thr_rows.append(dict(threshold=th, **m)); print(f"threshold {th}: ", fmt(m), f"cover={m['coverage']:.1f}%")
pd.DataFrame(thr_rows).round(3).to_csv(f"{RESULTS}/threshold_sweep.csv", index=False)

print("\n== Filters (pooled, threshold %.2f) ==" % THRESHOLD)
fil_rows = []
for f in FILTERS:
    m = pooled(keys, lambda s: (s["filt"][f], s["truth"])); fil_rows.append(dict(filter=f, **m)); print(f"{f:<16}", fmt(m))
pd.DataFrame(fil_rows).round(3).to_csv(f"{RESULTS}/filter_comparison.csv", index=False)

print("\n== Error by true elbow angle (does it under-read large angles?) ==")
X = np.concatenate([store[k]["base"] for k in keys]); Y = np.concatenate([store[k]["truth"] for k in keys])
ok = ~(np.isnan(X) | np.isnan(Y)); X, Y = X[ok], Y[ok]
edges = [0, 30, 60, 90, 120, 150, 181]; bin_rows = []
for a, b in zip(edges[:-1], edges[1:]):
    s = (Y >= a) & (Y < b)
    if s.sum() > 20:
        bin_rows.append(dict(true_angle=f"{a}-{min(b,180)}", n=int(s.sum()), mae=np.abs(X[s]-Y[s]).mean(), bias=(X[s]-Y[s]).mean()))
bins = pd.DataFrame(bin_rows).round(1); print(bins.to_string(index=False)); bins.to_csv(f"{RESULTS}/error_by_angle.csv", index=False)

# ---------- figures ----------
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
idx = np.random.default_rng(0).choice(len(X), min(6000, len(X)), replace=False)
mean_, diff_ = (X[idx] + Y[idx]) / 2, X[idx] - Y[idx]
ax[0].scatter(mean_, diff_, s=3, alpha=0.3)
ax[0].axhline((X - Y).mean(), color="k"); ax[0].axhline((X - Y).mean() + 1.96 * (X - Y).std(), color="r", ls="--"); ax[0].axhline((X - Y).mean() - 1.96 * (X - Y).std(), color="r", ls="--")
ax[0].set(title="Bland-Altman (MediaPipe - mocap)", xlabel="mean angle (deg)", ylabel="difference (deg)")
ax[1].bar(bins.true_angle, bins.bias); ax[1].axhline(0, color="k"); ax[1].set(title="Bias by true angle", xlabel="true elbow angle (deg)", ylabel="MediaPipe - mocap (deg)")
fd = pd.DataFrame(fil_rows); ax[2].bar(fd["filter"], fd["mae"]); ax[2].set(title="MAE by filter", ylabel="deg"); ax[2].tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.savefig(f"{RESULTS}/summary_figures.png", dpi=130); plt.show()

# example overlay: the recording with median r
ex_row = res.sort_values("r").iloc[len(res) // 2]; k = (ex_row.video, ex_row.side)
plt.figure(figsize=(11, 3.6)); plt.plot(store[k]["truth"], label="mocap (truth)"); plt.plot(store[k]["base"], label="MediaPipe")
plt.title(f"{k[0]} ({k[1]} arm), r={ex_row.r:.2f}, MAE={ex_row.mae:.1f}"); plt.xlabel("frame"); plt.ylabel("elbow angle (deg)"); plt.legend()
plt.tight_layout(); plt.savefig(f"{RESULTS}/example_overlay.png", dpi=130); plt.show()

## 9. Save everything and make the lines for `notes.md`

In [ ]:
p = pooled(keys, lambda s: (s["base"], s["truth"]))
print(f"""Paste into notes.md:

### Validation against motion capture (REHAB24-6, arm exercises, {len(res)} arm-recordings, threshold {THRESHOLD})
| Measure | Value |
|---|---|
| Frames compared | {p['n']} |
| MAE / RMSE | {p['mae']:.1f} / {p['rmse']:.1f} deg |
| Bias (MediaPipe - mocap) | {p['bias']:+.1f} deg |
| 95% limits of agreement | {p['loa_lo']:+.1f} to {p['loa_hi']:+.1f} deg |
| r / ICC(2,1) | {p['r']:.2f} / {p['icc']:.2f} |
""")
shutil.make_archive("/content/results_download", "zip", RESULTS)
try:
    from google.colab import files
    files.download("/content/results_download.zip")
except Exception as e:
    print("download manually from", RESULTS, "|", e)